<!---
Notas de clase Métodos Computacionales
Por 
Óscar Antonio Restrepo Gutiérrez
--->

# Solving linear systems
 
1. [Basic matrix operations with python](#Operaciones_matriciales).<br>
2. [Effect of multiplying a matrix by a vector](#Efecto_matriz_por_un_vector).<br>
3. [Solutions to systems of linear equations](#Soluciones_sistemas),<br>
   a. Inverse-matrix method.<br>
   b. Cramer's rule method.<br>
   c. Gaussian elimination.<br>
   d. Iterative Jacobi and Gauss-Seidel methods.<br>
4. [Matrix inverse and Gauss-Jordan](#Inversa_de_una_matriz).<br>
5. [Determinants](#Determinantes).<br> 
6. [Computational efficiency](#comparación_a_rutinas).<br>
7. [Supplementary material](#Complemento).<br>


<a id='Operaciones_matriciales'></a>
## Basic matrix operations with python

The routines for solving linear systems and linear algebra are implemented as methods in the `numpy` and `scipy` libraries. Scipy uses optimised libraries such as ATLAS, LAPACK and BLAS for [linear algebra](https://docs.scipy.org/doc/scipy/reference/tutorial/linalg.html); numpy also has its own libraries.
The libraries are imported as,

```python
import scipy.linalg as LA
import numpy.linalg as La
```

or also,

```python
from numpy import *       
from numpy.linalg import *
from scipy.linalg import * # for some functions such as lu, lu_solve, etc 
```

Or in a notebook you can simply use `%pylab`. 
To see the list of functions defined for linear algebra in scipy see this [link](https://docs.scipy.org/doc/numpy/reference/routines.linalg.html). Now let's use them:


In [ ]:
%pylab inline
from numpy.linalg import *

#### 1) Dimension of matrices 
A simple array is a matrix of dimension $n\times1$, or vector; for example the following numpy array of dimension 10,
```python
a = array([0,1,2,3,4,5,6,7,8,9])
```
Recall that `a == a[:] == a[0:10]`. To get a subarray the following notation is used,
```python
a[5:8] # gives the sub-vector array([5, 6, 7])      
```
On the other hand, matrices are built with two-dimensional arrays, for example the following $3\times 3$ matrix,
```python
A=array([[0,1,2],[3,4,5],[6,7,8]]) # array([[0, 1, 2],
                                   #        [3, 4, 5],
                                   #        [6, 7, 8]])
```
To get the individual elements you use `A[i,j]`, where $i$ runs over the rows and $j$ over the columns, thus,
```python
print( A[0,0], A[0,1],A[0,2] ) 
print( A[1,0], A[1,1],A[1,2] )
print( A[2,0], A[2,1],A[2,2] )
```
Note that `A[i,:]` gives row $i$ and `A[:,j]` gives column $j$. Example,

```python
A[1,:]   # row array([3, 4, 5])
A[:,1]   # column array([1, 4, 7])
A[1]     # same as A[1,:]
```
Note: if `A` is a 2D array, you can simply write `A[i]` to get row $i$.

#### 2) Submatrices
With the notation above submatrices can also be obtained, for example the submatrix,
([1, 2],[4, 5]), $i$ goes from $0$ to $2$ in the row and $j$ from $1$ to $3$ in the column, i.e.,
```python
A[0:2, 1:3] # gives array([[1, 2],
            #              [4, 5]])
# Also            
A[2, 1:3]   # gives array([7, 8])            
```
**Exercise**: create a $5\times 5$ matrix, and get its rows and columns individually, and generate submatrices.


In [ ]:
# Do the task
A = np.arange(25).reshape(5,5)
A


#### 3) Permuting rows and columns
To permute rows (or columns) you do it as follows,
```python
A[[0,2]] = A[[2,0]]
```
Permute columns 1 and 3
```python
A[:,[0,2]] = A[:,[2,0]]
```
**Exercise**: permute the previous matrix. 


In [ ]:
# Do the task


#### 4) Elementary operations
Addition and subtraction are done element by element,
```python
A = arange(9).reshape(3,3) # arange creates a 9-element array, reshape turns it into a 3x3 matrix.
B = array([[2,1,3],[-1,2,-1],[3,1,1]])
A+B # gives array([[2, 2, 5],
    #              [2, 6, 4],
    #              [9, 8, 9]])
    
A-B # gives array([[-2,  0, -1],
    #              [ 4,  2,  6],
    #              [ 3,  6,  7]])
```
In multiplication and division numpy "fails" since it multiplies element by element $a_{ij}*b_{ij}$ and $a_{ij}/b_{ij}$,
```python
A*B # gives array([[ 0,  1,  6], This is NOT matrix multiplication
    #              [-3,  8, -5],
    #              [18,  7,  8]])

A/B # gives array([[ 0,  1,  0], This is NOT matrix division or A*inv(B)
    #              [-3,  2, -5],
    #              [ 2,  7,  8]])
```
Numpy also has the `numpy.matrix()` instance to be able to do these operations,
```python
M1 = matrix([[1,2,3],[ 4,5, 6],[7,8,9]]) 
M2 = matrix([[2,1,3],[-1,2,-1],[3,1,1]])

M1 + M2 # gives the same as A + B  
M1 - M2 # gives the same as A - B
M1 * M2 # gives the matrix product of M1 and M2
M1 / M2 # gives the same as A/B
```
Note that `A/B` and `M1/M2` are not properly defined (these operations do element-wise division), but you can compute the inverse of the matrix created with array `A` using the `numpy.linalg.inv()` method,
```python
inv(A)  # A is a singular matrix (has no inverse, since a11 = 0 is zero)

inv(B)  # gives array([[-0.17647059, -0.11764706,  0.41176471],
        #              [ 0.11764706,  0.41176471,  0.05882353],
        #              [ 0.41176471, -0.05882353, -0.29411765]])
inv(M2) # gives the previous matrix, the same as the inverse of B but "matrix" instead of "array".
```
so matrix division is,
```python
M1*inv(M2)     # gives matrix([[ 1.29411765,  0.52941176, -0.35294118],
               #               [ 2.35294118,  1.23529412,  0.17647059],
               #               [ 3.41176471,  1.94117647,  0.70588235]])

```
In general, the matrix product of arrays can be done with several methods: `numpy.matmul(), numpy.dot` and the operator `@`,
```python
matmul(A,B)    # if A and B are 2D arrays
dot(A,B)       # if A and B are 2D arrays, and dot product if they're 1D arrays
A@B            # same as matmul or dot. 
```
these three methods are equivalent. 

You can turn an array into a matrix or vice versa,
```python
M1 = np.matrix(A) # given that A is defined as an array.
A  = np.array(M1) # given that M1 is defined as a Matrix.
```
**Exercise**: try out the previous operations on matrices `A, B, M1, M2`.


In [ ]:
# do the task


#### 5) Getting properties of a matrix
To compute the determinant use the `numpy.linalg.det()` method (note that M1 and $A$ are singular), 
```python
det(A)     # gives 0.0, matrix A is singular and has NO inverse.
det(M1)    # gives -9.5161973539299405e-16 this value is very small, so M1 has
           # no numerical inverse (on some machines inv(M1) may give a result
           # but with large numbers):
           #  matrix([[  3.15251974e+15,  -6.30503948e+15,   3.15251974e+15],
           #          [ -6.30503948e+15,   1.26100790e+16,  -6.30503948e+15],
           #          [  3.15251974e+15,  -6.30503948e+15,   3.15251974e+15]])
det(B) = 17.0
```
Transpose matrix of `M1`, note that $a_{ij}$ is swapped for $a_{ji}$,
```python
transpose(M1) #  numpy method
```
Getting a matrix's diagonal,
```python
diagonal(M1)  # gives array([1, 5, 9]), numpy method
```
The diagonal of a matrix,
```python
diag(A)       # gives array([1, 5, 9], numpy method
```
`numpy.diag(array, k)` is also used to create a diagonal matrix of dimension $(n+k)\times(n+k)$, i.e. with zeros and a diagonal at position `k` from the main diagonal (if `k` is positive the diagonal ends up on the upper part, otherwise on the lower part); let's see, 
```python
# matrix with one diagonal:
diag([3,3],2) # gives array([[0, 0, 3, 0],
              #              [0, 0, 0, 3],
              #              [0, 0, 0, 0],
              #              [0, 0, 0, 0]]))
# 5x5 matrix with 3 diagonals:
n = 5        # matrix dimension 
A = diag(arange(1,n+1))+diag(ones(n-2),2)+diag(ones(n-2),-2) 
```
Trace of the matrix, or sum of the diagonal elements.
```python
trace(M1)    # gives sum(array([1, 5, 9])) = 15

triu(M1)     # Upper triangular matrix
tril(M1)     # Lower triangular matrix
```
To get the dimension of the matrix, use,
```python
len(A)       # gives 3, number of rows of the matrix (for a vector it gives its dimension)
A.shape      # gives (3,3), the matrix's dimension.
```
#### 6) Solving matrix systems 
To solve the matrix system $A\mathbf{x} = \mathbf{b}$ by computing the inverse matrix: 
```python
b = array([1,1,-1])
x = matmul(inv(B),b) # Matrix multiplication if B and b are array type.

b = matrix([1,1,-1]) 
x = inv(M2)*b        # Matrix multiplication if M2 and b are matrix type.
```
Also $A\mathbf{x} = \mathbf{b}$ with the `numpy.solve()` method,
```python
x = solve(M2,b)      # More efficient than the previous method.
```
This last one is perhaps the most practical.
#### 7) Adding more rows or columns to a matrix
To add rows to a matrix the `numpy.r_[A,B,C,...]` command is used (note these are square brackets), or also `numpy.append()`; let's see,
```python
b = np.array([5,5,5])# creates a 4x3 matrix:
np.r_[A,[b]]         #  array([[0, 1, 2],
                     #         [3, 4, 5],
                     #         [6, 7, 8],
                     #         [5, 5, 5]]) 
np.r_[A,[b,b,b]]     # creates a 6x3 matrix
np.r_[A,A]           # creates a 6x3 matrix
append(A,b[None,:], axis=0)# adds a row 
append(A,A, axis=0)  # creates a 6x3 matrix (the two matrices must have equal dimension)
```
To add columns to a matrix the `numpy.c_[A,B,C,...]` method is used, for example,
```python
b = np.array([5,5,5])# creates a 3x4 matrix:
np.c_[A,b]           #  array([[0, 1, 2, 5],
                     #         [3, 4, 5, 5],
                     #         [6, 7, 8, 5]]) 
np.c_[A,b,b,b]       # creates a 3x6 matrix
np.c_[b]             # creates a column vector.
append(A,b[:,None], axis=1) # adds a column
append(A,A, axis=1)  # creates a 3x6 matrix (the two matrices must have equal dimension)
```
There are also the `np.hstack()` and `np.vstack()` commands, but it's easier to just remember the `np.r_[]` and `np.c_[]` commands.

#### 8) Copies of an array
Note that to make a copy of an array or matrix, doing `b = a` doesn't work,
since `b` only points to `a`'s memory address (`b` is an alias of `a`), and
modifying `b` modifies `a`; let's see,
```python
b = a                # a is array([0,1,2,3,4,5,6,7,8,9])
b[0] = 5
print(a[0])          # gives a[0] = 5
```
Better to create a real copy of `a`, use the `numpy.copy()` command,
```python
b = np.copy(a)       # where a is array([5,1,2,3,4,5,6,7,8,9])
b[0] = 0             # b is array([0,1,2,3,4,5,6,7,8,9])
print(a[0], b[0])    # gives a[0] = 5 and b[0] = 0
```
**Exercise**: verify these properties one by one.


In [ ]:
# Do the task:


In [ ]:
# Task: put the 3 diagonals together in this matrix
#       so it becomes a tridiagonal matrix
n = 5
A = diag(arange(1,n+1))+diag(2*ones(n-2),2)+diag(3*ones(n-2),-2)
A


<a id='Efecto_matriz_por_un_vector'></a>
## Effect of multiplying a matrix by a vector
When a vector is multiplied by a scalar number, the effect is to increase or decrease the vector's magnitude, i.e. it scales the vector (hence the word scalar); besides, if the number is negative it also changes its direction. But if the vector is multiplied by a matrix, the effect is - besides the change in magnitude - a change in direction, which in some cases means a rotation (this can also be interpreted as a change of basis according to linear algebra); to see this let's use [arrow](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.annotate.html#matplotlib.pyplot.annotate) to plot the vectors,
```python
matplotlib.pyplot.arrow(x, y, dx, dy, \*\*kwargs)
```
which plots a vector from $(x, y)$ to $(x+dx, y+dy)$.


In [ ]:
scalar = -.5
x, y = 4, 2  # vector with coordinates x,y
arrow(0, 0, x, y, head_width=0.2, head_length=.2, fc='k', ec='k')

#------ vector times scalar --------------------------
v1 = scalar*array([x,y]) # scaled vector to plot
arrow(0, 0, v1[0], v1[1], head_width=0.2, head_length=.2, fc='r', ec='r')

#------  Matrix times vector --------------------------
#A = np.random.uniform(-1,1,size=(2,2)) # random 2x2 matrix
A = np.random.rand(2,2) # random 2x2 matrix
print(A)
v2 = A@v1               # multiply matrix by vector
arrow(0, 0, v2[0], v2[1], head_width=0.2, head_length=.2, fc='b', ec='b')

ylim(-5,5)
xlim(-5,5)
grid()


In general, consider the matrix $A \equiv A(\theta)$, which depends on the angle $\theta$ and defines a [rotation](https://en.wikipedia.org/wiki/Rotation_matrix) about the axis passing through the origin of coordinates and perpendicular to the plane of rotation (for example the angular velocity),

$$
A(\theta) =
\begin{bmatrix}
\cos \theta & -\sin \theta \\[3pt]
\sin \theta & \cos \theta \\
\end{bmatrix},
$$

any vector multiplied by this matrix will rotate by an angle $\theta$; let's see:


In [ ]:
def A(theta):
    return np.array([[np.cos(theta),-np.sin(theta)],
                      [np.sin(theta), np.cos(theta)]])

theta = -np.pi/2     
v1 = np.array([3,0])  # unrotated vector
v2 = A(theta)@v1      # vector rotated by an angle theta

figure(1,figsize = (5,5))
arrow(0, 0, v1[0], v1[1], head_width=0.2, head_length=.2, fc='r', ec='r')
arrow(0, 0, v2[0], v2[1], head_width=0.2, head_length=.2, fc='b', ec='b')

ylim(-4,4) # range in y
xlim(-4,4) # range in x
grid()


**Task**: implement a clock where one arrow is the minute hand and another the second hand, so that given a time (min,sec) it puts the arrows in the indicated place.

**Task**: the cross product can be seen as an antisymmetric matrix formed with the elements of vector $\mathbf{a}$, which when multiplied by a vector $\mathbf{b}$ gives another vector at $90^\circ$ degrees relative to the original vectors, 

$$\mathbf{a} \times \mathbf{b} = [\mathbf{a}]_{\times} \mathbf{b} = \begin{bmatrix}\,0&\!-a_3&\,\,a_2\\ \,\,a_3&0&\!-a_1\\-a_2&\,\,a_1&\,0\end{bmatrix}\begin{bmatrix}b_1\\b_2\\b_3\end{bmatrix}.$$

Using this definition, write a python program that, given the position $\mathbf {r}$ and velocity $\mathbf {v}$, computes a) the angular velocity defined as, 

$$\boldsymbol\omega=\frac{\mathbf r\times\mathbf v}{r^2}=\omega \mathbf {u} ={\frac {d\phi }{dt}}\mathbf {u} ={\frac {v\sin(\theta )}{r}}\mathbf {u},$$

where $\theta$ is the angle between $\mathbf{r}$ and $\mathbf{v}$, and $\phi$ and $\mathbf {u}$ are the angle and axis of rotation of the body. b) Compute the body's tangential velocity given by,

$$\mathbf{v}_{\perp} =\boldsymbol{\omega} \times\mathbf{r}.$$
<!---
Note that the angular velocity omega can be seen as the matrix
$$
\Omega =[\omega ]_{\times }={\begin{bmatrix}\,\,0&\!-\omega _{3}&\,\,\,\omega _{2}\\\,\,\,\omega _{3}&0&\!-\omega _{1}\\\!-\omega _{2}&\,\,\omega _{1}&\,\,0\end{bmatrix}}.
$$
--->


In [ ]:
# Do the task:


<a id='Soluciones_sistemas'></a>
## Solving systems of linear equations

Linear systems are common in physics; for example Kirchhoff's laws in circuits generate linear systems of the form,

$$ a_{11}x_1 + a_{12}x_2 + \cdots a_{1n}x_n = b_1 $$
$$\vdots$$
$$ a_{n1}x_1 + a_{n2}x_2 + \cdots a_{nn}x_n = b_n $$

There are several ways to solve linear systems of equations: by matrix inversion, by Cramer's rule and by Gaussian elimination; the latter is of special interest given its easy computational implementation. Below is a brief description of each:

### a) Matrix inversion
The first method is by matrix inversion; if $A\mathbf{x} = \mathbf{b}$ then,

$$A^{-1}A \mathbf{x} = A^{-1}\mathbf{b} $$ 

where $A^{-1}A = \mathbf{I} \rightarrow  \mathbf{x} = A^{-1}\mathbf{b}$

### b) Cramer's rule 
to solve the system $A\mathbf{x} = \mathbf{b}$

$$      x_i = \frac{\det A(\mathbf{a}_i\rightarrow \mathbf{b})}{\det A} $$ 

where $i=b$ indicates that column $i$ of matrix $A$ is replaced by vector $b$,
example in a system of 2 equations:

$$
x_1 = \frac{
\begin{vmatrix}
b_1& a_{12}\\ b_2& a_{22}
\end{vmatrix}
}{
\begin{vmatrix}
a_{11}& a_{12}\\ a_{21}& a_{22}
\end{vmatrix}
}
\quad\hbox{ and }\quad
x_2 = \frac{
\begin{vmatrix}
a_{11}&b_1\\ a_{21}&b_2
\end{vmatrix}
}{
\begin{vmatrix}
a_{11}& a_{12}\\ a_{21}& a_{22}
\end{vmatrix}
}
$$

### c) Gaussian elimination by back substitution

In a system of equations, let $E_i$ with $i=1,2,...,n,$ be each of the $n$ equations; then
the following operations can be performed on each $E_i$
 
  a. multiply by a factor $b: bE_i → E_i$ <br>
  b. add or subtract another equation multiplied by $b: E_i + bE_j → E_i$ <br>
  c. permute the order of two equations: $E_i \leftrightarrow E_j$. <br>
 
Let's start with an example,

$$
\begin{array}{r}
E_1 :&  x_1 &+&  x_2 & &      &+& 3x_4 &=& 4,\\
E_2 :& 2x_1 &+&  x_2 &−&  x_3 &+&  x_4 &=& 1,\\
E_3 :& 3x_1 &−&  x_2 &−&  x_3 &+& 2x_4 &=&−3,\\
E_4 :& −x_1 &+& 2x_2 &+& 3x_3 &−&  x_4 &=& 4,\\
\end{array}
$$

if we subtract $(E_2 -2E_1) → E_2, (E_3 -3E_1) → E_3, (E_4 + E_1) → E_4$, then,

$$
\begin{array}{r}
E_1 :&  x_1&+&  x_2 & &      &+& 3x_4 &=&  4,\\
E_2 :&     &-&  x_2 &-&  x_3 &-& 5x_4 &=&- 7,\\
E_3 :&     &-& 4x_2 &-&  x_3 &-& 7x_4 &=&-15,\\
E_4 :&     &+& 3x_2 &+& 3x_3 &+& 2x_4 &=&  8,\\
\end{array}
$$

then $(E_3 − 4E_2) → E_3$ and $(E_4 + 3E_2) → E_4$,

$$
\begin{array}{r}
E_1 :&  x_1 &+&  x_2 & &      &+&  3x_4 &=&  4,\\
E_2 :&      &-&  x_2 &-&  x_3 &-&  5x_4 &=& -7,\\
E_3 :&      & &      & & 3x_3 &+& 13x_4 &=& 13,\\        
E_4 :&      & &      & &      &-& 13x_4 &=&-13,\\
\end{array}       
$$

Finally we see that $x_4 = 1$; we substitute into $E_3$ to find $x_3 = 0$, and into $E_2$ to get $x_2 = 2$, and from $E_1, x_1 = -1$,


In general, consider the system of equations $E_i$,
 
$$
\begin{array}{r} 
E_1 :& a_{11}x_1 &+& a_{12}x_2 &+&\cdots&+& a_{1n}x_n &=& b_1,\\
E_2 :& a_{21}x_1 &+& a_{22}x_2 &+&\cdots&+& a_{2n}x_n &=& b_2,\\
\vdots&          & &           & &\ddots& &           & &      \\
E_n :& a_{n1}x_1 &+& a_{n2}x_2 &+&\cdots&+& a_{nn}x_n &=& b_n,\\
\end{array}
$$

create the augmented matrix,
 
$$
\left( \matrix{
a_{11}& a_{12}&...&a_{1n}&\vdots& a_{1,n+1}\\
a_{21}& a_{22}&...&a_{2n}&\vdots& a_{2,n+1}\\
&&\ddots&\\
a_{n1}& a_{n2}&...&a_{nn}&\vdots& a_{n,n+1}\\
}\right) 
$$

where $a_{i,n+1} = b_i$ for each $i = 1, 2, · · · , n.$

If $a_{11}$ is nonzero, we do the operations, 
 
$$(E_j − (a_{j1}/a_{11})E_1) → (E_j) \quad\hbox{for each}\quad j = 2, 3, . . . , n$$

and then the operations (given that $a_{ii} \neq 0$),

$$(E_j − (a_{ji}/a_{ii})E_i) → (E_j) \quad\hbox{for each}\quad j = i + 1, i + 2, . . ., n,$$
       
This iterative procedure gives the triangular matrix,

$$
\left( \matrix{
 a_{11}& a_{12} &\cdots& a_{1n}&\vdots& a_{1,n+1}\\
       & a_{22} &\cdots& a_{2n}&\vdots& a_{2,n+1}\\
       &        &\ddots&        &\\
       &        &   & a_{nn}&\vdots& a_{n,n+1}\\
}\right) 
$$

and the new, equivalent algebraic system has the new triangular form,

$$
\begin{array}{r}
        E_1 &:& a_{11}x_1 + a_{12}x_2 + \cdots+a_{1n}x_n &=& a_{1,n+1},\\
        E_2 &:&           + a_{22}x_2 + \cdots+a_{2n}x_n &=& a_{2,n+1},\\
       &       & \ddots \quad\quad\quad&&\\
        E_n &:&                              a_{nn}x_n &=& a_{n,n+1},\\
\end{array} 
$$


clearly $x_n$ is,

$$x_n = a_{n,n+1}/a_{nn}$$

and the next term $x_{n-1}$ is given by,

$$x_{n−1} = ( a_{n−1,n+1} − a_{n−1,n}x_n )/a_{n−1,n−1}.$$
       
So in general, the rest of the $x_i$ are obtained by back substitution, i.e.,

$$
\begin{eqnarray}
        x_i &=& \frac{a_{i,n+1} − a_{in}x_n − a_{i,n−1}x_{n−1} − · · · −a_{i,i+1}x_{i+1}}{a_{ii}},\\
            &=& \frac{a_{i,n+1} − \sum_{j=i+1}^na_{ij}x_j}{a_{ii}},\\
\end{eqnarray} 
$$



for $i = n − 1, n − 2, · · · , 2, 1$.

The Gaussian elimination method is fairly easy to implement in computing and is the standard for solving linear systems; in fact, this is the method implemented in `linalg.solve(A,b)`. 

**Exercise**: Implement Gaussian elimination (Hint: note from the equations that to build the triangular matrix you need two nested `for` loops, one with $i=1,...,n$ and the other with $j=i+1,...,n$; once the matrix is built you need another `for` with $i=1,...,n$ to get the $x_i$. For the pseudocode you can see Burden's book, page 364). Compare the result against the one obtained with the `linalg.solve(A,b)` routine.


In [ ]:
# Solution: 
# Implementation of the Gaussian elimination routine.

def Solve_Gauss(A,b):
    '''
    Solves the system, Ax = b, by Gaussian elimination.
    '''
    n = len(A)                   # Number of rows of A.
    M = np.c_[A,b]               # Create augmented matrix.
    
    #------- Build the triangular matrix: ---------------------------
    for i in range(n):           # Loop over the rows.
                        
        for j in range(i+1,n):   # Operation Ej - b*Ei -> Ej.
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 
                
    #------- Solve x from the triangular matrix: -------
    x = np.zeros(n)
    x[n-1] = M[n-1,n]/M[n-1,n-1] # Compute x_n
    for i in range(n-1,-1,-1):   # Compute x_i with i=n−1,···,2,1
        x[i] = (M[i,n] - sum(M[i,i+1:n]*x[i+1:n]))/M[i,i]

    return M, x   # Return the triangular matrix and vector x.


A, b = np.random.random((4,4)), np.random.random(4)
D, x = Solve_Gauss(A,b)
D, x, solve(A,b) # Compared against np.linalg.solve()


### d) Iterative Jacobi and Gauss-Seidel methods
A fourth way to solve the system $A\mathbf{x} = \mathbf{b}$ is by iterative methods; for this $A$ is split as the sum of its diagonal matrix and the rest, ${\mathit {A}}={\mathit {D}}+{\mathit {R}}$, so $({\mathit {D}}+{\mathit {R}})\mathbf{x} = \mathbf{b}$, which gives 
$D\mathbf {x}=\left(\mathbf {b} -{\mathit {R}}\mathbf {x}\right)$, which (by the fixed-point method, since $\mathbf {x}=\mathbf {g}(\mathbf {x})=D^{-1}\left(\mathbf {b} -{\mathit {R}}\mathbf {x}\right)$, but in vector form) lets us define the *Jacobi* method by the iteration,

$$
\begin{eqnarray}
\mathbf {x} ^{\left(k+1\right)}&=&{\mathit {D}}^{-1}\left(\mathbf {b} -{\mathit {R}}\mathbf {x} ^{(k)}\right)\\
x_{i}^{\left(k+1\right)}&=&{\frac {1}{a_{ii}}}\left(b_{i}-\sum^n \limits _{j\neq i}{a_{ij}x_{j}^{\left(k\right)}}\right),\quad k=1,2,3,...
\end{eqnarray}
$$

Any vector can be used as the initial approximation $\mathbf{x}^0$, but ideally it should be close to $\mathbf{x}$; it converges if $A$ is diagonally dominant, i.e. if $\left|a_{ii}\right|>\sum _{j\neq i}{\left|a_{ij}\right|}$.

The *Gauss-Seidel* method is defined by decomposing $A$ as $A=L+U$, where $L$ is the lower triangular decomposition with $A$'s main diagonal and $U$ is upper triangular with zeros on the diagonal; then, $A\mathbf{x} = (L+U)\mathbf{x}=\mathbf{b}$ and rearranging, $L\mathbf{x}=\mathbf{b}- U\mathbf{x}$, which gives the iterative procedure,

$$
\mathbf {x} ^{(k+1)}=L^{-1}(\mathbf {b}-U\mathbf {x} ^{(k)}).
$$

if we take advantage of the triangular shape of $L$, the elements $x_i^{k+1}$ can be computed sequentially, so the sequence is,

$$
x_{i}^{(k+1)}={\frac {1}{a_{ii}}}\left(b_{i}-\sum _{j=1}^{i-1}a_{ij}x_{j}^{(k+1)}-\sum _{j=i+1}^{n}a_{ij}x_{j}^{(k)}\right),\quad k=1,2,3,\dots. 
$$
The pseudocode is,
```julia
function gauss_seidel(A, b, x, kmax)
    for k = 1:kmax
        for j = 1:size(A)
            x[j] = (1/A[j,j])*(b[j] - A[j,:]*x[j] + A[j,j]*x[j])
        end
    end
end
```
This method is faster than Jacobi and always converges when $A$ is diagonally dominant or symmetric and positive-definite. In general, any iterative method that can be written in the form $\mathbf {x}^{(k+1)}=T\mathbf {x}^k+\mathbf {c}$ converges if the largest eigenvalue of $T$ is smaller than one, i.e. $|\lambda_{max}|<1.$

**Task**: implement the Jacobi and Gauss-Seidel routines.


In [ ]:
# do the task


<a id='Inversa_de_una_matriz'></a>
# Matrix inverse and Gauss-Jordan
The inverse of a non-singular matrix can be computed by several methods; two of these are: by cofactors (i.e. via determinants, and, as we'll see later, very expensive computationally) and by Gaussian elimination — the latter is implemented via Gauss-Jordan elimination.

## Gauss-Jordan method
Consider the system $A\mathbf{x} = \mathbf{b}$; the Gauss-Jordan method consists of applying Gaussian elimination until $A$ is transformed into the identity matrix, so that $\mathbf{b}$ is transformed into $\mathbf{x}$; it's useful for solving several systems of linear equations at the same time, i.e., suppose you need to compute the solution to the following systems,

$$
A\mathbf{x}_1 = \mathbf{b}_1,\\ 
A\mathbf{x}_2 = \mathbf{b}_2,\\
\vdots\\
A\mathbf{x}_m = \mathbf{b}_m,
$$

then the matrix $B\equiv B(\mathbf{b}_1,\mathbf{b}_2,...,\mathbf{b}_m)$ is created with columns $\mathbf{b}_i$ and the matrix $X\equiv X(\mathbf{x}_1,\mathbf{x}_2,...,\mathbf{x}_m)$ with columns $\mathbf{x}_i$, giving the system $AX=B$, which has solution $X = A^{-1}B$; this is solved by Gaussian elimination: create the augmented matrix of $A$ and $B$ as, $M \equiv [A\vdots B]_{n,n+m}$, and apply Gaussian elimination until the $A$ side of $M$ becomes the identity matrix; this process transforms the $B$ side into $X$. 
Note that if $B = I$ is the identity matrix then $X = A^{-1}$, i.e. this method also serves to compute the matrix inverse; let's see, consider the product $AX = I$, then,

$$ AA^{-1} = AX= \left( \matrix{
a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
       &        & \ddots &        \\
a_{m1} & a_{n2} & \cdots & a_{nn} 
}\right)\left( \matrix{
x_{11} & x_{12} & \cdots & x_{1n} \\
x_{21} & x_{22} & \cdots & x_{2n} \\
       &        & \ddots &        \\
x_{n1} & x_{n2} & \cdots & x_{nn} 
}\right) = 
\left( \matrix{
1 & 0 & \cdots & 0 \\
0 & 1 & \cdots & 0 \\
  &   & \ddots &   \\
0 & 0 & \cdots & 1
}\right)$$


if we use the definition of the augmented matrix, this can be solved as,

$$M=\left( \matrix{
a_{11} & a_{12} & \cdots & a_{1n} & \vdots & 1 & 0 & \cdots & 0 \\
a_{21} & a_{22} & \cdots & a_{2n} & \vdots & 0 & 1 & \cdots & 0 \\
       &        & \ddots &        &        &   &   & \ddots &\\
a_{n1} & a_{n2} & \cdots & a_{nn} & \vdots & 0 & 0 & \cdots & 1
}\right),$$

and by Gauss-Jordan elimination the matrix $M$ is converted into $M'$, where matrix $A$ becomes the identity matrix, and on the right-hand side of $M'$ we obtain matrix $X$ with coefficients $x_{ij}$, 

$$M'=\left( \matrix{
1 & 0 & \cdots & 0 & \vdots & x_{11} & x_{12} & \cdots & x_{1n} \\
0 & 1 & \cdots & 0 & \vdots & x_{21} & x_{22} & \cdots & x_{2n} \\
       &        & \ddots &        &        &   &   & \ddots &\\
0 & 0 & \cdots & 1 & \vdots & x_{n1} & x_{n2} & \cdots & x_{nn}
}\right).$$

### Building the Gauss-Jordan routine
To build the Gauss-Jordan routine, note that in the Gaussian elimination routine equation $E_i$ is used to eliminate the lower terms $a_{ji}$ of equations $E_j$ with $j=i+1, ...,n$; in this case equation $E_i$ must also be used to eliminate the upper terms $a_{ji}$ with $j=1, ...,i-1$ (i.e., in the previous Gaussian elimination routine add another `for j in range(0,i)`); this technique is known as *Gauss-Jordan*. 

Finally, from this calculation we see it is more expensive to compute the inverse than to directly solve the system $A\mathbf{x} = \mathbf{b}$ by Gaussian elimination.

**Exercise**: write a routine that computes the inverse by Gaussian elimination with Gauss-Jordan, solving the system $AX=B$. 


In [ ]:
# Do the task
def Solve_Gauss_Jordan(A,B):
    '''
    Solves the system, AX = B, by Gaussian elimination.
    '''
    n = len(A)                   # Number of rows of A.
    M = np.c_[A,B]               # Create augmented matrix.
    
    #------- Build the triangular matrix: ---------------------------
    for i in range(n):           # Loop over the rows.
 
        for j in range(0,i):     # Operation Ej - b*Ei -> Ej.
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 

        for j in range(i+1,n):   # Operation Ej - b*Ei -> Ej.
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 
                
    #------- Solve x from the diagonal matrix: -------
    for i in range(0,n):   
          M[i] = M[i]/M[i,i]

    return M[:,n:]    # Return triangular matrix and vector x.

n = 4
A, b = np.random.random((n,n)), np.random.random(n)
B = np.eye(n,n)
X = Solve_Gauss_Jordan(A,B)
X, solve(A,B), inv(A) # Compared against np.linalg: solve() and inv()


Alternatively, note that the above is equivalent to using a single `for` in `j` with `i != j`, i.e. we don't touch the diagonal:


In [ ]:
# More compact alternative. 
def Solve_Gauss_Jordan(A,B):
    '''
    Solves the system, AX = B, by Gaussian elimination.
    '''
    n = len(A)                   # Number of rows of A.
    M = np.c_[A,B]               # Create augmented matrix.
    
    for i in range(n):           # Loop over the rows.
        for j in range(n):       # Operation Ej - b*Ei -> Ej.
            if i != j:
                M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 
                        
    #------- Solve x from the diagonal matrix: -------
    for i in range(0,n):   
          M[i] = M[i]/M[i,i]

    return M[:,n:]    # Return triangular matrix and vector x.

n = 4
A, b = np.random.random((n,n)), np.random.random(n)
B = np.eye(n,n)
X = Solve_Gauss_Jordan(A,B)
X, solve(A,B), inv(A) # Compared against np.linalg: solve() and inv()


<a id='Determinantes'></a>
# Determinant of a matrix
The determinant is a scalar property of square matrices and is useful for computing the solution of linear systems of equations; for instance, such a system has a solution if its determinant is non-zero; it's also useful for computing a matrix's inverse.




## Computing determinants

Let $A$ be an $n\times n$ matrix, with coefficients $a_{ij}$; then:
1. If $n=1$, $A$ is a scalar $a$ and its determinant is $\det A = a$,
2. If $n>1$, the minor $\det(M_{ij})$ is defined as the determinant of the $(n-1)\times(n-1)$ matrix obtained by removing row $i$ and column $j$ from $A$.
3. The cofactor $C_{ij}$ associated with the determinant of $M_{ij}$ is defined as $C_{ij} = (-1)^{i+j}\det(M_{ij})$.
4. In general, the determinant of an $n\times n$ square matrix $A$ is computed as,

$$ \det A = \sum_{j=1}^n a_{ij}C_{ij} \quad\hbox{ for all }\quad i=1,...,n.$$

or also,

$$ \det A = \sum_{i=1}^n a_{ij}C_{ij} \quad\hbox{ for all }\quad j=1,...,n.$$

given this, the inverse of $A$ is computed as, 

$$A^{-1}= \frac{1}{\det{A}}C^T,$$

where $C^T$ is the transpose of the matrix defined by the cofactors $C_{ij}$ (known as the adjugate of $A$).
Note that the determinant can be computed along any row or column; it's advisable to use the one with the most zeros. As seen in the mathematical formula, the implementation can be done recursively in python (since the determinant depends on the minor, which is itself another determinant), fixing $i=1$ (any $i$ can be chosen according to the formula in 4. above), i.e.,

$$\det A = \sum_{j=1}^n (-1)^{j}a_{1j} \det(M_{1j})$$

The implementation is (remember that in python $i$ starts at 0, not 1):


In [ ]:
import numpy as np
from numpy.linalg import det    

# Determinant by cofactors, recursively with i=0
# Very slow, proportional to t ~ n!
def Det(A):
    n = len(A)

    if n==1:
        return A[0,0]
    else:
        d = 0
        sign = -1
        for j in range(n):
            sign = -sign                  # Cofactor sign
            k = np.delete(range(n),j)     # Remove index j
            d += sign*A[0,j]*Det(A[1:,k]) # Cofactor
        return d
    
A=np.random.random((5,5))    
Det(A), det(A)            # Compared against linalg.det()  


### Properties of determinants
From the definition of the alternating multilinear form ([see supplement](#forma_multilineal_alternada)) the following properties can be proven, 

1. If all elements of $A$ are $a_{ij}=0$, $\det A = 0$.
2. If a row or column of $A$ is all zeros, $\det A = 0$.
3. If $\hat A$ is obtained by the permutation $(E_i)\leftrightarrow (E_j)$, ($i\neq j$) then $\det \hat A=-\det A$.
4. If $\hat A$ is obtained by the rescaling $(\lambda E_i)\leftrightarrow (E_i)$, then $\det \hat A=\lambda \det A$.
5. If $\hat A$ is obtained by the substitution $(E_i+\lambda E_j) \leftrightarrow (E_i)$, ($i\neq j$) then $\det \hat A=\det A$.
6. $\det(AB)=(\det A)(\det B).$
7. $\det A^t=\det A.$
8. $\det A^{-1}=(\det A)^{-1}$
9. If $A$ is an upper (or lower) triangular matrix,

$$\det A = \prod_{i=1}^n a_{ii} $$
 
The previous algorithm takes a time of $n!$, which is prohibitive for matrices with large $n$, but if the matrix is triangular the computation is quite simple and only requires $n$ multiplications, according to property 9. Therefore, the previous properties can be used together with Gaussian elimination to transform matrix $A$ into a triangular one, and this only costs $~n^3$ additional operations.
 
 
**Example**: in eight operations matrix $A$ is transformed into the upper-triangular matrix $A_8$, i.e.,

$$
A=
\left(\begin{array}{r}
2 & 1 & -1 & 1\\
1 & 1 & 0  & 3\\
-1& 2 & 3  &-1\\
3 &−1 & -1 & 2
\end{array}\right)
\quad\Longrightarrow\quad A_8=
\left(\begin{array}{r}
1 &\frac{1}{2}& -\frac{1}{2}& \frac{1}{2}\\
0 &1 & 1 & 5   \\
0 &0 & 3 & 13  \\
0 &0 & 0 & -13
\end{array}\right)
$$

and the determinant of the triangular matrix is -39, and can be computed as, 

$
\begin{array}{l}
1.& \hbox{The operation } (\frac{1}{2} E_1      \rightarrow     E_1)& \hbox{ gives }& \det A_1=\frac{1}{2}\det A\\
2.& \hbox{The operation } (E_2 - E_1            \rightarrow     E_2)& \hbox{ gives }& \det A_2= \det A_1=\frac{1}{2}\det A\\
3.& \hbox{The operation } (E_3 + E_1            \rightarrow     E_3)& \hbox{ gives }& \det A_3= \det A_2=\frac{1}{2}\det A\\
4.& \hbox{The operation } (E_4 - 3E_1           \rightarrow     E_4)& \hbox{ gives }& \det A_4= \det A_3=\frac{1}{2}\det A\\
5.& \hbox{The operation } (2E_2                 \rightarrow     E_2)& \hbox{ gives }& \det A_5=2\det A_4=\det A\\
6.& \hbox{The operation } (E_3 - \frac{5}{2}E_2 \rightarrow     E_3)& \hbox{ gives }& \det A_6= \det A_5=\det A\\
7.& \hbox{The operation } (E_4 + \frac{5}{2}E_2 \rightarrow     E_4)& \hbox{ gives }& \det A_7= \det A_6=\det A\\
8.& \hbox{The operation } (E_3                  \leftrightarrow E_4)& \hbox{ gives }& \det A_8=−\det A_7=−\det A\\
\end{array}
$

finally the determinant is 39.

**Task**: how many determinants do you need to compute to get the matrix inverse? How many operations does this take?

**Task**: create a random matrix and verify the properties of determinants.

**Task**: with python, show that if $A$ is triangular its inverse is triangular and has determinant $\det(A^{-1})=1/\det A$.

**Exercise**: implement the computation of determinants via Gaussian elimination and the properties above. 




In [ ]:
# Do the task
def Solve_Gauss(A,b):
    '''
    Solves the system, Ax = b, by Gaussian elimination.
    '''
    n = len(A)                   # Number of rows of A.
    M = np.c_[A,b]               # Create augmented matrix.
    
    #------- Build the triangular matrix: ---------------------------
    for i in range(n):           # Loop over the rows.
                        
        for j in range(i+1,n):   # Operation Ej - b*Ei -> Ej.
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 
    
    #------- Determinant of the triangular matrix: --------------------
    Det=1        
    for i in range(n):
        Det *= M[i,i]
        
    #------- Solve x from the triangular matrix: -------
    x = np.zeros(n)
    x[n-1] = M[n-1,n]/M[n-1,n-1] # Compute x_n
    for i in range(n-1,-1,-1):   # Compute x_i with i=n−1,···,2,1
        x[i] = (M[i,n] - sum(M[i,i+1:n]*x[i+1:n]))/M[i,i]

    return Det, M, x  # Return triangular matrix and vector x.


A, b = np.random.random((4,4)), np.random.random(4)
det(A), Solve_Gauss(A,b), solve(A,b) # Compared against np.linalg.solve()


<a id='comparación_a_rutinas'></a>
# Computational efficiency
Computational efficiency refers to two things: first, the error in matrix calculations, and second, the computation time of these operations. If there are no approximations, the error is due to rounding of arithmetic operations, and the computation time will depend on the number of these operations: additions/subtractions and multiplications/divisions. Since computation times for additions/subtractions are different from multiplications/divisions, they should be computed separately (the difference will depend on whether the numbers are integers - addition may be faster - or floats - multiplication may be faster -, on the compiler used and on the [processor architecture](http://nicolas.limare.net/pro/notes/2014/12/12_arit_speed/), so no single value can be given). Let's first look at the effects of rounding:

### Rounding errors in Gaussian elimination
Although the Gaussian elimination method is exact, rounding the coefficients to a given number of significant digits has a computational cost that shows up in the arithmetic operations; to better understand this, consider the following example given in Burden's book,

$$
\begin{eqnarray}
E_1 &:& 0.003x_1 &+ 59.14x_2 &= 59.17\\
E_2 &:& 5.291x_1 &- 6.130x_2 &= 46.78 
\end{eqnarray}
$$

If only four significant figures are considered, 
and the operation $(E_2 + b E_1)\rightarrow (E_2)$ is done, where 

$$b = -\frac{a_{21}}{a_{11}} = -\frac{5.291}{0.003} = 1763.666\cdots \approx 1764$$

If we round to four figures (for example in the second equation, $-104329.09 \approx -104300$), we have,

$$
\begin{eqnarray}
E_1 &:& 0.003x_1 &+ 59.14x_2 &=& 59.17\\
E_2 &:&          &-104300x_2 &=& -104400 
\end{eqnarray}
$$

which gives the wrong solution, 

$$
x_1 \approx \frac{59.17 - (59.14)(1.001)}{0.00300} = -10\quad\text{ and }\quad
x_2 \approx 1.001
$$

but if all digits are used,

$$
\begin{eqnarray}
E_1 &:& 0.003x_1& + 59.14x_2            & = & 59.17 \\
E_2 &:& 0       &  -104309.37\bar{6}x_2 & = & -104309.37\bar{6}
\end{eqnarray}
$$

and the exact solution is,

$$ x_1 = 10.00 \quad\text{ and }\quad x_2 = 1.00.$$

The error occurs because $59.14/0.00300 \approx 20000$, which propagates through the operations giving the wrong result — i.e. if $a_{ii}$ is very small, or worse, zero, the result will be disastrous. One way to minimise this type of error is to permute rows (or columns) to place the largest coefficient $|a_{ji}|$ in the position of $|a_{ii}|$, such that the equation used for the operations $(E_j − (a_{ji}/a_{ii})E_i → E_j)$ always has the largest coefficient $|a_{ii}|$ among all possible equations $E_j$; in other words, among the equations with $j=i,i+1,...,n$, we choose the equation $E_j$ with the largest coefficient $|a_{ji}|$ and swap it with $E_i$, i.e. we do,

$$ |a_{ii}| = \max_{i\leq j\leq n}|a_{ji}|$$

This technique is known as the *pivoting method*. For example, in the previous problem we simply swap equations $E_1$ and $E_2$,

$$
\begin{eqnarray}
E_2 &:& 5.291x_1 &- 6.130x_2 &= 46.78,\\ 
E_1 &:& 0.003x_1 &+ 59.14x_2 &= 59.17,\\
\end{eqnarray}
$$

if the problem is redone with only four significant figures it gives the correct answer, since the divisor is now $5.291$. 

Normally pivoting is only done on rows; if done on rows and columns the number of operations and computation time increase.

**Exercise**: implement row pivoting in your Gaussian routine to reduce rounding error.


In [ ]:
# do the task


### Efficiency in matrix multiplication
In the addition/subtraction of two $n\times n$ matrices $(A+B)$, $n^2$ operations are needed (since each matrix has $n^2$ elements), so the time is obviously proportional to $O(n^2)$. However, for matrix multiplication $AB$, $n^2$ dot products (row $i$ times column $j$) must be computed, since for each row and column $n$ multiplications and $(n-1)$ additions are needed, so the number of operations is given by $n^2(n+(n-1))=2n^3-n^2 = O(n^3)$; as we'll see later, some techniques let us reduce this time down to order $O(n^2)$.


### Efficiency of Gaussian elimination
 It can be shown (Burden page 366) that the number of multiplications and divisions required in Gaussian elimination is,

$$\frac{n^3}{3}+n^2-\frac{n}{3}$$

and the number of additions and subtractions is given by,

$$\frac{n^3}{3}+\frac{n^2}{2}-\frac{5n}{6}$$

As can be seen, for large $n$ the dominant term is $n^3/3$, i.e. time scales as $t\sim O(n^3)$.
For Gauss-Jordan it can be shown that the number of multiplications and divisions is, 
$$\frac{n^3}{2}+n^2-\frac{n}{2}$$

and the number of additions and subtractions,

$$\frac{n^3}{2}-\frac{n}{2}$$


**Task**: build matrices with $n=10,100, 500$ and compute the computation times for the product and Gaussian elimination.


In [ ]:
# do the task


### Computation times in matrix calculations
Using our own linear algebra routines is not the most efficient approach, since efficiency depends on how the arithmetic operations are carried out and on the programming language; it's best to use already implemented and tested libraries. Let's compare the timings of the `scipy.linalg` library against the routines built here in python, which should be done statistically; first let's see how to make a histogram:

#### How to make histograms
To generate a histogram on the interval $[a,b]$ with $M$ bins, consider,

$$\Delta = \frac{b-a}{M}\\ i=\text{int}\left(\frac{x-a}{\Delta}\right)$$

so, every time a value falls between $x$ and $x+\Delta$, it is stored in $h[i]$ as,

$$h[i]=h[i]+1$$

that is, $h[i]$ counts all the values generated between $x$ and $x+\Delta$; you could implement the code, but the `plt.hist(x,bins)` method does this for us (
this method computes $\Delta$ as $\frac{x_{max}-x_{min}}{bins}$)


In [ ]:
h=[1,3,4,5,4,3,3,1,3,3,3,1,3,5,4,4,4,1,2,3,6]
# histogram with 6 default bins
hist(h)#, bins=2) # use 2 or 4 bins


**Exercise**: for a random ($20\times20$) matrix, compute the average computation time for the Gauss routine implemented in python and the `scipy.solve(A,b)` routine; compare also against `matmul(inv(A),b)` where `b = np.random.rand(20)`, and build the histogram of computation times for $500$ runs.


In [ ]:
# Compute average time
from datetime import datetime

Nrep = 500 # Number of repetitions
n = 20     # Matrix dimension M

def Calcular_Tiempo(n, Nrep):

    Times = np.zeros(Nrep)      # Initialise array to zero
    for i in range(Nrep):

        #M = np.array(np.random.random((n,n+1)))
        M = np.array(np.random.random((n,n)))
        b = np.random.random(n)
    
        tstart = datetime.now() # Start time
        # Timing comparison
        # Gaussian_Elimination(M) # Copy from Bustamante's page
        Solve_Gauss(M,b)  # 1)
        #inv(M)@b          # 2)
        #matmul(inv(M),b)   # 3) 
        #solve(M,b)          # 4) 
 
        tend = datetime.now()  # End time
        
        Times[i] = (tend-tstart).microseconds # Save time difference in array
        
    
    print ("Average time %lf microseconds"%(Times.mean()))
    
    #--- Histogram --------
    plt.figure( figsize=(8,5) )
    plt.hist( Times, bins = 30 )
    plt.xlabel( "t μs" )
    plt.ylabel( "Occurrences" )
    plt.grid()
    
    return Times.mean()
        
Calcular_Tiempo( n, Nrep )


<a id='Complemento'></a>
# Supplementary material


### Concatenating matrices: adding rows and columns to a matrix
There are several ways to concatenate two matrices or add rows and columns:
`np.append()`, `np.vstack()` and `np.hstack()`, but the easiest is to use the `np.r_[A,B,C,...]` and `np.c_[A,B,C,...]` commands for rows and columns; they are used as follows:



In [ ]:
import numpy as np
A=np.array([[1,2,3],[4,5,6],[7,8,9]])
b=np.array([1,1,1])

# Add rows 
np.c_[A,b], np.c_[A,b,b,b,A]


In [ ]:
# Add columns (can also be done with np.append())
np.r_[A,[b]], np.r_[A,[b],[b],[b],A]


### $n$-dimensional arrays
Consider the following situation for a $3\times 4$ matrix: suppose you want the sum of each of its rows, but `A.sum()` gives the sum of all $12$ elements; to do this, use `axis`,
```python
A.sum(axis=0) # gives an array with the column sums
A.sum(axis=1) # gives an array with the row sums
```
In general for arrays with more than one index ([$n$-dimensional arrays](https://docs.scipy.org/doc/numpy/reference/arrays.ndarray.html)), you should use `axis=n` to define along which dimension the method is applied, i.e. it collapses the operation over all elements in dimension $n$ (see [link](https://stackoverflow.com/questions/17079279/how-is-axis-indexed-in-numpys-array)); this applies to most methods defined in numpy: `A.prod()`, `A.mean()`, `A.std()`, `max()`, `min()`,`argmax()`, `argmin()`, etc. Important: when using axis, please check that the answer is along the desired dimension.

**Task**: try `axis` with different methods:


In [ ]:
A = array([[1,1,1,1],
           [2,2,2,4],
           [3,3,3,8]])

A.sum(axis=0), A.sum(axis=1) # sum columns and rows
#A.prod(axis=0), A.prod(axis=1)
#A.mean(axis=0), A.mean(axis=1)


### Gaussian elimination routine with pivoting
The following routine does Gaussian elimination with row pivoting to reduce rounding error.
Other routines are on the [Rosettacode](https://rosettacode.org/wiki/Gaussian_elimination#Julia) page. Note that after pivoting we check that `M[i,i]` is not zero; if it is, the matrix is singular.


In [ ]:
def Solve_Gauss(A,b):
    '''
    Solves the system, Ax = b, by Gaussian elimination.
    '''
    n = len(A)                   # Number of rows of A.
    M = np.c_[A,b]               # Create augmented matrix.
    
    #------- Build the triangular matrix: ---------------------------
    for i in range(n):           # Loop over the rows.
        
        jmax = i                 # Index of the largest Mji.                                            
        for j in range(i,n):     # Pivot on rows.          
            if abs(M[j,i]) > abs(M[i,i]): jmax = j 
                
        M[[i,jmax]] = M[[jmax,i]]# Swap rows i, jmax.

        if M[i,i]==0.0:          # Check that the matrix is non-singular.
            print("Error, this matrix is singular.")
            return None, None
                
        for j in range(i+1,n):   # Operation Ej - b*Ei -> Ej.
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 
                
    #------- Solve x from the triangular matrix: -------
    x = np.zeros(n)
    x[n-1] = M[n-1,n]/M[n-1,n-1] # Compute x_n
    for i in range(n-1,-1,-1):   # Compute x_i with i=n−1,···,2,1        
        x[i] = (M[i,n] - sum(M[i,i+1:n]*x[i+1:n]))/M[i,i]

    return M, x   # Return triangular matrix and vector x.

# M = np.matrix( np.random.random((4,5)))# random matrix
# D, x = Solve_Gauss(M[:,:-1],M[:,-1])   # fails if M is matrix()
A, b = np.random.random((4,4)), np.random.random(4)
#A = np.arange(16).reshape(4,4)*1.0 # singular matrix
A[0,0] = 0. # fails without pivoting.
D, x = Solve_Gauss(A,b)
D, x, solve(A,b) # Compared against np.linalg.solve()


### Gauss-Jordan elimination routine 
Recall that the system $AA^{-1}=AX=I$ can be written as a system of $n$ equations of the form,

$$ \left( \matrix{
a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
\vdots & \vdots & & \vdots\\
a_{n1} & a_{n2} & \cdots & a_{nn} 
}\right)\left( \matrix{
x_{11} \\
x_{21} \\
\vdots \\
x_{n1}
}\right) = 
\left( \matrix{
1 \\
0 \\
\vdots \\
0
}\right),$$

$$\left( \matrix{
a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
\vdots & \vdots & & \vdots\\
a_{n1} & a_{n2} & \cdots & a_{nn} 
}\right)\left( \matrix{
x_{12} \\
x_{22} \\
\vdots \\
x_{n2}
}\right) = 
\left( \matrix{
0 \\
1 \\
\vdots \\
0
}\right),$$

$$\vdots$$ 
$$\left( \matrix{
a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
\vdots & \vdots & & \vdots\\
a_{n1} & a_{n2} & \cdots & a_{nn} 
}\right)\left( \matrix{
x_{1n} \\
x_{2n} \\
\vdots \\
x_{nn}
}\right) = 
\left( \matrix{
0 \\
0 \\
\vdots \\
1
}\right).
$$

This system can be solved individually by Gaussian elimination, but the following routine uses the definition of the augmented matrix.

The following routine does Gaussian elimination with the Gauss-Jordan method (without pivoting — please implement it).



In [ ]:
def Solve_Gauss_Jordan(A,B):
    '''
    Gauss-Jordan method
    Solves the system, AX = B, by Gaussian elimination,
    where B is a matrix or a vector; if B = I is the 
    identity matrix, it computes the inverse of A.
    '''
    n = len(A)                  # Number of rows of A.
    M = np.c_[A,B]               # Create augmented matrix.
    
    #------- Build the triangular matrix: ---------------------------
    for i in range(n):          # Loop over the rows (equations Ei).
        
        #=========================================
        # Task: implement row pivoting here
        #=========================================
        
        for j in range(0,i):    # Operation Ej - b*Ei -> Ej to put
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] # zeros in the upper triangle. 
            
        for j in range(i+1,n):  # Operation Ej - b*Ei -> Ej to put
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] # zeros in the lower triangle. 
       
    for i in range(n):
        M[i,n:] = M[i,n:]/M[i,i]# (B side) get X as M[:,n:] and
        M[i,i] = M[i,i]/M[i,i]  # (A side) get the identity matrix. 
        # M[i] = M[i]/M[i,i]    # Replaces the previous lines but uses 
                                # more time. REMEMBER M[i] is row i.       

    return M[:,n:]              # return X (drop the identity matrix). 


A, b = np.random.random((4,4)), np.random.random(4)
X = Solve_Gauss_Jordan(A,b)
X, solve(A,b) # Compared against np.linalg.solve()


In [ ]:
# Computing the matrix inverse with Gauss-Jordan.
A = np.random.random((4,4))
I = np.eye(4) # identity matrix

X = Solve_Gauss_Jordan(A,I)
X, inv(A) # Compared against np.linalg.inv()


In [ ]:
solve(A,eye(4)) # solve(A,B) also works


### Comparing average computation time


In [ ]:
# Comparison of the average computation time of the solution AX=b
# for the methods: 
#             1) np.solve, 
#             2) inv(M)*b,
#             3) matmul(M,b) 
#             4) Solve_Gauss(M,b)  

# %matplotlib qt
from datetime import datetime

Nrep = 5000 # Number of repetitions
n = 20      # Matrix dimension M

def Calcular_Tiempo(n, Nrep):

    Times = np.zeros((4,Nrep))      # Initialise array to zero
    for i in range(Nrep):

        #M = np.array(np.random.random((n,n+1)))
        M = np.array(np.random.random((n,n)))
        b = np.random.random(n)
        I = eye(n)              # identity matrix
        
        tstart = datetime.now() # Start time
        solve(M,b)          # 1) 
        tend = datetime.now()   # End time
        Times[0,i] = (tend-tstart).microseconds # Save time difference in array

        tstart = datetime.now() # Start time
        inv(M)@b            # 2)
        tend = datetime.now()   # End time
        Times[1,i] = (tend-tstart).microseconds # Save time difference in array
 
        tstart = datetime.now() # Start time
        matmul(inv(M),b)    # 3) 
        tend = datetime.now()   # End time
        Times[2,i] = (tend-tstart).microseconds # Save time difference in array
 
        tstart = datetime.now() # Start time
        Solve_Gauss(M,b)   # 4)   very slow.
        tend = datetime.now()   # End time
        Times[3,i] = (tend-tstart).microseconds # Save time difference in array
       
    meanT = Times.mean(axis=1) # average time per row
    #--- Histograms (note that Δ = (x.max-x.min)/bins = (3000)/500 = 6 ) --------
    plt.figure( figsize=(8,5) )
    plt.hist( Times[0], bins = 500, label='%7.3lf μs, np.solve(M,b)'%(meanT[0]),range=(0,3000) )
    plt.hist( Times[1], bins = 500, label='%7.3lf μs, np.inv(M)@b'%(meanT[1])  ,range=(0,3000) )
    plt.hist( Times[2], bins = 500, label='%7.3lf μs, matmul(inv(M),b)'%(meanT[2]) ,range=(0,3000) )
    plt.hist( Times[3], bins = 500, label='%7.3lf μs, Solve_Gauss(M,b)'%(meanT[3]),range=(0,3000) )

    plt.xlabel( "t μs" )
    plt.ylabel( "Occurrences" )
    plt.grid()
    plt.legend()

print ("Average times, microseconds (μs)")
Calcular_Tiempo( n, Nrep )


In [ ]:
# Comparison of the average computation time for the inverse of A, 
# via the solution AX=B, for the methods:
#   1) inv(M)
#   2) solve(M,I)
#   3) Solve_Gauss_Jordan(M,I)

from datetime import datetime

Nrep = 5000 # Number of repetitions
n = 20     # Matrix dimension M

def Calcular_Tiempo(n, Nrep):

    Times = np.zeros((3,Nrep))      # Initialise array to zero
    for i in range(Nrep):

        #M = np.array(np.random.random((n,n+1)))
        M = np.array(np.random.random((n,n)))
        b = np.random.random(n)
        I = eye(n)              # identity matrix
        
        tstart = datetime.now() # Start time
        inv(M)                 
        tend = datetime.now()   # End time
        Times[0,i] = (tend-tstart).microseconds # Save time difference in array

        tstart = datetime.now() # Start time
        solve(M,I)         
        tend = datetime.now()   # End time
        Times[1,i] = (tend-tstart).microseconds # Save time difference in array
 
        tstart = datetime.now() # Start time
        Solve_Gauss_Jordan(M,I) # very slow.
        tend = datetime.now()   # End time
        Times[2,i] = (tend-tstart).microseconds # Save time difference in array
    
    meanT = Times.mean(axis=1) # average time per row
    #--- Histograms (note that Δ = (x.max-x.min)/bins = (3000)/500 = 6 ) --------
    plt.figure( figsize=(8,5) )
    plt.hist( Times[0], bins = 500, label='%7.3lf μs, inv()     '%(meanT[0]),range=(0,3000) )
    plt.hist( Times[1], bins = 500, label='%7.3lf μs, solve()   '%(meanT[1]),range=(0,3000) )
    plt.hist( Times[2], bins = 500, label='%7.3lf μs, Solve_GJ()'%(meanT[2]),range=(0,3000) )

    plt.xlabel( "t μs" )
    plt.ylabel( "Occurrences" )
    plt.grid()
    plt.legend()

print ("Average times, microseconds (μs)")
Calcular_Tiempo( n, Nrep )


<a id='forma_multilineal_alternada'></a>
### Note on determinants

In general the determinant is defined as an alternating multilinear form, i.e., let $A=f(\mathbf{a_1},...,\mathbf{a_n})$ be a matrix as a function of its columns, $\mathbf{a_i}$; then,

$$
f(\mathbf{a}_1,...,\lambda\mathbf{a}_i+\mathbf{b}_i,...,\mathbf{a}_n)
=\lambda f(\mathbf{a}_1,...,\mathbf{a}_i,...,\mathbf{a}_n)
+f(\mathbf{a}_1,...,\mathbf{b}_i,...,\mathbf{a}_n).\\
f(\mathbf{a}_1,...,\mathbf{a}_i,...,\mathbf{a}_j,...,\mathbf{a}_n)
=-f(\mathbf{a}_1,...,\mathbf{a}_j,...,\mathbf{a}_i,...,\mathbf{a}_n)
$$

The first property defines linearity and the second the alternation with respect to sign.

In geometric terms the determinant defines the oriented volume of $n$-dimensional space; so, for example, for $3D$ space, the absolute value of the determinant gives the volume of the parallelepiped defined by the three vectors, $\mathbf{a},\mathbf{b},\mathbf{c}$, thus,

$$
\det A(\mathbf{a},\mathbf{b},\mathbf{c})=\begin{vmatrix} a_1 & b_1 & c_1\\ a_2 & b_2 & c_2\\ a_3 & b_3 & c_3
\end{vmatrix}.
$$

Note that if $A$ is a matrix with coefficients generated with a uniform distribution, then as $n\rightarrow \infty, \det(A)\rightarrow 0$.
Some additional routines for determinants:


In [ ]:
# Fast way to compute the determinant (though linalg.det(A) is 
# faster still) by Gaussian elimination with row pivoting.

def Det_gauss(A):
    '''
    Computes the determinant by Gaussian elimination.
    '''
    n = len(A)                   # Number of rows of A.
    M = np.copy(A)               # Create augmented matrix.
    
    #------- Build the triangular matrix: ---------------------------
    sign = 1.
    for i in range(n):           # Loop over the rows.
        
        jmax = i                 # Index of the largest Mji, for
        for j in range(i+1,n):   # pivoting on rows.          
            if abs(M[j,i]) > abs(M[i,i]): 
                jmax = j 

        M[[i,jmax]] = M[[jmax,i]]# Swap rows i, jmax.
        if jmax != i: sign = -sign # Flip sign if there's a permutation.
        if M[i,i] == 0.0: return 0.0 # check whether the matrix is singular.
            
        for j in range(i+1,n):   # Operation Ej - b*Ei -> Ej.
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 
 
    Det = 1.                                         
    for i in range(n): 
        Det *= M[i,i]            # determinant of triangular matrix

    return sign*Det

A = np.random.random((100,100))
A = np.arange(16).reshape(4,4)*1.0# singular matrix, A must be float.
Det_gauss(A), det(A)             # compared against linalg.det() 


In [ ]:
%timeit Det_gauss(A)


In [ ]:
# Fast way to compute the determinant (though linalg.det(A) is 
# faster still) by Gaussian elimination with row pivoting.
# Better alternative than the previous one thanks to numba.
from numba import njit

@njit
def Det_gauss(A):
    '''
    Computes the determinant by Gaussian elimination with row pivoting.
    '''
    n = len(A)                   # Number of rows of A.
    M = np.copy(A)               # Create augmented matrix.
    
    #------- Build the triangular matrix: ---------------------------
    sign = 1.
    for i in range(n):           # Loop over the rows.
        
        jmax = i                 # Index of the largest Mji, for
        for j in range(i+1,n):   # pivoting on rows.          
            if abs(M[j,i]) > abs(M[i,i]): 
                jmax = j 

#        M[[i,jmax]] = M[[jmax,i]]# Swap, doesn't work with numba.
        swap = np.copy(M[i,:])   # Swap rows i and imax 
        M[i,:] = M[jmax,:]
        M[jmax,:] = swap

        if jmax !=i: sign = -sign# Flip sign if there's a permutation.
        if M[i,i] == 0.0: return 0.0 # check whether the matrix is singular.       
        
        for j in range(i+1,n):   # Operation Ej - b*Ei -> Ej.
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 
 
    Det = 1.                                         
    for i in range(n): 
        Det *= M[i,i]            # determinant of triangular matrix

    return sign*Det

A = np.random.random((100,100))
Det_gauss(A), det(A)             # compared against linalg.det() 


In [ ]:
# Fast way to compute the determinant by Gaussian elimination.
# Fastest option if numba is used, but less precise since it only pivots if M[i,i]=0.
# https://integratedmlai.com/find-the-determinant-of-a-matrix-with-pure-python-without-numpy-or-scipy/

from numba import njit
@njit
def Det_fast(A):
    # Section 1: Establish n parameter and copy A
    n = len(A)
    M = np.copy(A)
    sign = 1. 
    # Section 2: Row ops on A to get in upper triangle form
    for i in range(n):           # A) i stands for focus diagonal
        for j in range(i+1,n):   # B) only use rows below i row
            if M[i,i] == 0:      # C) if diagonal is zero ...
#                M[i,i] = 1.0e-10 # change to ~zero( but NOT GOOD IDEA IF n IS LARGE), then:

                # swap row i with row jmax (this is pivoting technique):
                jmax = np.argmax(np.abs(M[i:,i])) + i # get the row index of the largest elem

                swap = np.copy(M[i,:]) # do permutation of row i with row jmax: 
                M[i,:] = M[jmax,:]
                M[jmax,:] = swap
            
                sign = -sign     # change sign after permutation
            
            Scal = M[j,i]/M[i,i] # D) cr stands for "current row"            
            for k in range(n):   # E) cr - Scal * iRow, one element at a time
                M[j,k] = M[j,k] - Scal*M[i,k]
     
    # Section 3: Once M is in upper triangle form ...
    prod = 1.0
    for i in range(n):        
        prod *= M[i,i]           # product of diagonals is determinant

    return sign*prod


A = np.random.random((100,100))
A[0,0]= 0.         # case 1, not good when M[i,i] = 1.0e-10 and worse if M[i,i] = 1.0e-18
#A[0,0] =  1.0e-15 # case 2, the two functions differs
#A=np.array([[0,1.],[1.,4.]])#

#A=np.array([[0.003,59.14],[5.291,-6.130]])# example of pivoting
Det_fast(A), det(A)# Compared against linalg.det()


In [ ]:
# algorithm 2) (very slow, since it's by cofactors)
# from 
# https://stackoverflow.com/questions/47465356/how-to-find-determinant-of-matrix-using-python
# Warning: In Python accessing a "nested list" cannot be done by multi-dimensional slicing, 
# i.e.: A[0,0], instead one would write A[0][0], this is because m is a list and not an array.

import numpy as np

def determinant(A, mul=1):
 d = len(A)
 if d == 1:
    return mul*A[0][0] 
 else:
    sign = -1
    sum = 0
    for i in range(d):
        m = []
        for j in range(1, d):
            buff = [] 
            for k in range(d):
                if k != i:
                    buff.append(A[j][k])
            m.append(buff)
        sign *= -1
        sum += mul*determinant(m, sign*A[0][i])# here we introduce a list, not an array
    return sum

A = np.array([[1,-2,3],[0,-3,-4],[0,0,-3]])
#A=np.random.random((10,10))# takes long time! 
determinant(A)


In [ ]:
# Timing comparison for the three determinant routines

%timeit Det_fast(A) # very fast with numba (only pivots if Mii=0), the slowest without numba
%timeit Det_gauss(A)# 3 times slower due to pivoting 
%timeit det(A)      


In [ ]:
# Compute the most common element in an array
a=np.array([0,0,0,0,1,1,2,2,2,2,2,2,2,1,1,1,1,1])
np.argmax(np.bincount(a))
#scipy.stats.mode(a) # this is slower
#np.bincount(a)


In [ ]:
# Jacobi's method; to guarantee convergence A must 
# be diagonally dominant: |a_ii| > ∑_j≠i |a_ij|

import numpy as np

def jacobi(A, b, x0, eps=1e-10, maxiter=500): 
    D = np.diag(np.diag(A))  # Matrix with A's diagonal.
    R = A - D                # Subtract the diagonal.
    xold = x0 
    for i in range(maxiter): 
        Dinv = np.diag(1./np.diag(D)) # Inverse of D.
        xnew = np.dot(Dinv, b-np.dot(R, xold)) 
        dx = xnew-xold
        if np.dot(dx, dx) < eps**2.: break
        xold = xnew 
    return xnew 
 
# A must be diagonally dominant (remove the diag and see what happens):
A = np.random.rand(4,4) + np.diag([2,2,2,2])
b = np.array([9, 3, 5, 1]) 
 
x0 = np.zeros(4) 
x = jacobi(A, b, x0) 
 
x, np.linalg.solve(A,b) 


### Note on computer arithmetic 
Is multiplication slower or faster than addition on modern CPUs?

*In integer arithmetic*, addition is usually appreciably faster. It has been observed differences of the order of 3 times faster, more for 8-byte objects. 

*In real arithmetic*, multiplication may be faster for the following reason:
When two real numbers are multiplied, the mantissae are multiplied together and the exponents are added, and these operations can be carried out in parallel. When two real numbers are added, first the mantissa of the smaller number must be shifted so that the exponents match (a process termed normalisation). Then the mantissae must be added. The result of the addition may overflow the original word length by 1 bit, or it may generate any number of leading zeros. Therefore the result must be normalised again. There are therefore 3 steps and they must be done in series.
